# Course Similarity Model Comparison

We need a course similarity model for several tasks: recommending courses similar to ones the student is interested in, finding alternatives when a course doesn't fit, and scoring the interest factors in our decision model.

We compare four approaches:
1. **TF-IDF:** Bag-of-words baseline on course descriptions. Fast, no model needed, but misses semantic relationships.
2. **Sentence Embeddings:** `all-MiniLM-L6-v2` on course descriptions. Captures semantic similarity ("neural networks" ↔ "deep learning").
3. **Hybrid:** Sentence embeddings blended with structured features (department, prerequisites, units).
4. **LLM Direct:** Ask the LLM directly for similar courses. Tests whether baked-in knowledge beats structured approaches.

**Test:** For 3 anchor courses, show the top 5 most similar courses under each model. Check whether results match intuition.

In [1]:
# ── Setup ──
import json, os, sys, re
import numpy as np
from src.config import BASE_MODEL, MY_MODEL, HF_TOKEN, DATA_DIR
from huggingface_hub import InferenceClient

# Load courses
with open(os.path.join(DATA_DIR, "courses.json"), "r") as f:
    COURSES_LIST = json.load(f)
COURSES = {c["subject_id"]: c for c in COURSES_LIST}

client = InferenceClient(model=MY_MODEL or BASE_MODEL, token=HF_TOKEN)

# Filter to courses with descriptions (needed for text-based models)
courses_with_desc = {cid: c for cid, c in COURSES.items() 
                     if c.get("description") and len(c["description"]) > 20}
print(f"Total courses: {len(COURSES)}")
print(f"Courses with descriptions: {len(courses_with_desc)}")

# Anchor courses for testing
ANCHORS = ["6.3900", "18.06", "14.01"]
for cid in ANCHORS:
    c = COURSES.get(cid, {})
    print(f"\n  {cid} — {c.get('title', '?')}")
    print(f"    {c.get('description', '')[:150]}...")

Total courses: 6808
Courses with descriptions: 6800

  6.3900 — Introduction to Machine Learning
    Introduction to the principles and algorithms of machine learning from an optimization perspective. Topics include linear and non-linear models for su...

  18.06 — Linear Algebra
    Basic subject on matrix theory and linear algebra, emphasizing topics useful in other disciplines, including systems of equations, vector spaces, dete...

  14.01 — Principles of Microeconomics
    Introduces microeconomic concepts and analysis, supply and demand analysis, theories of the firm and individual behavior, competition and monopoly, an...


---
## Model 1: TF-IDF

Represent each course by its description as a TF-IDF vector, then compute cosine similarity between the anchor and all other courses. This is a pure keyword-matching baseline — it will find courses that share the same words but miss semantic relationships.

In [3]:
# ── Model 1: TF-IDF similarity ──
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Build corpus: course descriptions
cids = list(courses_with_desc.keys())
descriptions = [courses_with_desc[cid].get("description", "") for cid in cids]

# Fit TF-IDF
tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf_matrix = tfidf.fit_transform(descriptions)

# Build index
cid_to_idx = {cid: i for i, cid in enumerate(cids)}

def tfidf_similar(anchor_cid, top_n=5):
    if anchor_cid not in cid_to_idx:
        return []
    idx = cid_to_idx[anchor_cid]
    sims = cosine_similarity(tfidf_matrix[idx:idx+1], tfidf_matrix)[0]
    # Sort and exclude self
    ranked = sorted(enumerate(sims), key=lambda x: x[1], reverse=True)
    results = []
    for i, score in ranked:
        if cids[i] != anchor_cid:
            results.append((cids[i], score))
        if len(results) >= top_n:
            break
    return results

print("MODEL 1: TF-IDF")
print("=" * 80)
for anchor in ANCHORS:
    title = COURSES[anchor]["title"]
    print(f"\n  Anchor: {anchor} — {title}")
    for cid, score in tfidf_similar(anchor):
        t = COURSES[cid]["title"]
        dept = cid.split(".")[0]
        print(f"    {score:.3f}  {cid:12s} {t[:50]}  (dept {dept})")

MODEL 1: TF-IDF

  Anchor: 6.3900 — Introduction to Machine Learning
    0.345  18.415       Advanced Algorithms  (dept 18)
    0.345  6.5210       Advanced Algorithms  (dept 6)
    0.341  6.7900       Machine Learning  (dept 6)
    0.338  6.8200       Sensorimotor Learning  (dept 6)
    0.310  15.680       Machine Learning: Algorithms, Applications, and Co  (dept 15)

  Anchor: 18.06 — Linear Algebra
    0.967  ES.1806      Linear Algebra  (dept ES)
    0.370  18.700       Linear Algebra  (dept 18)
    0.363  2.087        Engineering Mathematics: Linear Algebra and ODEs  (dept 2)
    0.358  18.0651      Matrix Methods in Data Analysis, Signal Processing  (dept 18)
    0.355  18.C06       Linear Algebra and Optimization  (dept 18)

  Anchor: 14.01 — Principles of Microeconomics
    0.266  11.003       Methods of Policy Analysis  (dept 11)
    0.266  17.303       Methods of Policy Analysis  (dept 17)
    0.248  10.491       Integrated Chemical Engineering II  (dept 10)
    0.226  1.266 

---
## Model 2: Sentence Embeddings

Use `sentence-transformers/all-MiniLM-L6-v2` to embed each course description into a 384-dimensional vector, then compute cosine similarity. This captures semantic relationships — "optimization" and "gradient descent" should score high even without shared keywords.

In [5]:
# ── Model 2: Sentence embeddings ──
from sentence_transformers import SentenceTransformer

# Load model (runs on CPU, ~80MB)
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Embed all descriptions
print("Embedding course descriptions...")
embeddings = embed_model.encode(descriptions, show_progress_bar=True, batch_size=64)
print(f"Embeddings shape: {embeddings.shape}")

def embedding_similar(anchor_cid, top_n=5):
    if anchor_cid not in cid_to_idx:
        return []
    idx = cid_to_idx[anchor_cid]
    anchor_emb = embeddings[idx:idx+1]
    sims = cosine_similarity(anchor_emb, embeddings)[0]
    ranked = sorted(enumerate(sims), key=lambda x: x[1], reverse=True)
    results = []
    for i, score in ranked:
        if cids[i] != anchor_cid:
            results.append((cids[i], score))
        if len(results) >= top_n:
            break
    return results

print("\nMODEL 2: Sentence Embeddings")
print("=" * 80)
for anchor in ANCHORS:
    title = COURSES[anchor]["title"]
    print(f"\n  Anchor: {anchor} — {title}")
    for cid, score in embedding_similar(anchor):
        t = COURSES[cid]["title"]
        dept = cid.split(".")[0]
        print(f"    {score:.3f}  {cid:12s} {t[:50]}  (dept {dept})")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding course descriptions...


Batches:   0%|          | 0/107 [00:00<?, ?it/s]

Embeddings shape: (6800, 384)

MODEL 2: Sentence Embeddings

  Anchor: 6.3900 — Introduction to Machine Learning
    0.723  6.862        Applied Machine Learning  (dept 6)
    0.646  6.7900       Machine Learning  (dept 6)
    0.634  6.7250       Optimization for Machine Learning  (dept 6)
    0.592  15.680       Machine Learning: Algorithms, Applications, and Co  (dept 15)
    0.569  6.8200       Sensorimotor Learning  (dept 6)

  Anchor: 18.06 — Linear Algebra
    0.982  ES.1806      Linear Algebra  (dept ES)
    0.707  18.0651      Matrix Methods in Data Analysis, Signal Processing  (dept 18)
    0.698  18.335       Introduction to Numerical Methods  (dept 18)
    0.698  6.7310       Introduction to Numerical Methods  (dept 6)
    0.696  18.065       Matrix Methods in Data Analysis, Signal Processing  (dept 18)

  Anchor: 14.01 — Principles of Microeconomics
    0.727  15.010       Economic Analysis for Business Decisions  (dept 15)
    0.726  15.0111      Economic Analysis for Busi

---
## Model 3: Hybrid (Embeddings + Structured Features)

Blend the sentence embedding similarity with structured features: same department (+0.1), shared prerequisites (+0.05 per shared prereq), and similar unit count (+0.05 if within 3 units). Tests whether structural signals improve over pure text similarity.

In [6]:
# ── Model 3: Hybrid similarity ──

def get_prereq_set(cid):
    """Extract prerequisite course IDs from a course's prereq string."""
    course = COURSES.get(cid, {})
    prereq_str = course.get("prerequisites", "")
    if not prereq_str or prereq_str.lower() == "none":
        return set()
    return set(re.findall(r'([\d]+\.[\w.]+|[A-Z]+\.[\w.]+)', prereq_str))

def structural_bonus(anchor_cid, other_cid):
    """Compute structural similarity bonus between two courses."""
    a = COURSES.get(anchor_cid, {})
    b = COURSES.get(other_cid, {})
    bonus = 0.0
    
    # Same department
    dept_a = anchor_cid.split(".")[0]
    dept_b = other_cid.split(".")[0]
    if dept_a == dept_b:
        bonus += 0.10
    
    # Shared prerequisites
    prereqs_a = get_prereq_set(anchor_cid)
    prereqs_b = get_prereq_set(other_cid)
    shared = len(prereqs_a & prereqs_b)
    bonus += shared * 0.05
    
    # Similar units
    units_a = a.get("total_units", 12)
    units_b = b.get("total_units", 12)
    if abs(units_a - units_b) <= 3:
        bonus += 0.05
    
    return bonus

def hybrid_similar(anchor_cid, top_n=5, text_weight=0.7, struct_weight=0.3):
    if anchor_cid not in cid_to_idx:
        return []
    idx = cid_to_idx[anchor_cid]
    anchor_emb = embeddings[idx:idx+1]
    text_sims = cosine_similarity(anchor_emb, embeddings)[0]
    
    results = []
    for i, cid in enumerate(cids):
        if cid == anchor_cid:
            continue
        text_score = text_sims[i]
        struct_score = structural_bonus(anchor_cid, cid)
        combined = text_weight * text_score + struct_weight * struct_score
        results.append((cid, combined, text_score, struct_score))
    
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_n]

print("MODEL 3: Hybrid (Embeddings + Structure)")
print("=" * 80)
for anchor in ANCHORS:
    title = COURSES[anchor]["title"]
    print(f"\n  Anchor: {anchor} — {title}")
    for cid, combined, text_s, struct_s in hybrid_similar(anchor):
        t = COURSES[cid]["title"]
        dept = cid.split(".")[0]
        print(f"    {combined:.3f}  {cid:12s} {t[:45]}  (text={text_s:.2f} struct={struct_s:.2f})")

MODEL 3: Hybrid (Embeddings + Structure)

  Anchor: 6.3900 — Introduction to Machine Learning
    0.551  6.862        Applied Machine Learning  (text=0.72 struct=0.15)
    0.512  6.7900       Machine Learning  (text=0.65 struct=0.20)
    0.504  6.7250       Optimization for Machine Learning  (text=0.63 struct=0.20)
    0.448  6.7910       Statistical Learning Theory and Applications  (text=0.55 struct=0.20)
    0.443  6.8200       Sensorimotor Learning  (text=0.57 struct=0.15)

  Anchor: 18.06 — Linear Algebra
    0.703  ES.1806      Linear Algebra  (text=0.98 struct=0.05)
    0.540  18.0651      Matrix Methods in Data Analysis, Signal Proce  (text=0.71 struct=0.15)
    0.533  18.335       Introduction to Numerical Methods  (text=0.70 struct=0.15)
    0.532  18.065       Matrix Methods in Data Analysis, Signal Proce  (text=0.70 struct=0.15)
    0.523  18.338       Eigenvalues of Random Matrices  (text=0.68 struct=0.15)

  Anchor: 14.01 — Principles of Microeconomics
    0.524  15.010  

---
## Model 4: LLM Direct

Ask the LLM directly: "What MIT courses are most similar to X?" This tests whether the model's training data knowledge produces good results without any structured computation. The risk is hallucination — the LLM might suggest courses that don't exist.

In [7]:
# ── Model 4: LLM direct similarity ──

LLM_SIMILARITY_PROMPT = """You are an MIT course advisor. Given a course, list the 5 most similar MIT courses.

For each similar course, provide the course number, title, and a brief reason why it's similar.

Respond with EXACTLY this format:
1. [course_number] — [title]: [reason]
2. [course_number] — [title]: [reason]
...

IMPORTANT: Only suggest real MIT courses. Use official course numbers (e.g., 6.3900, 18.06)."""

print("MODEL 4: LLM Direct")
print("=" * 80)

llm_results = {}
for anchor in ANCHORS:
    title = COURSES[anchor]["title"]
    desc = COURSES[anchor].get("description", "")[:200]
    
    messages = [
        {"role": "system", "content": LLM_SIMILARITY_PROMPT},
        {"role": "user", "content": f"Course: {anchor} — {title}\nDescription: {desc}"},
    ]
    
    try:
        response = client.chat_completion(messages=messages, max_tokens=300, temperature=0.1)
        raw = response.choices[0].message.content.strip()
        
        print(f"\n  Anchor: {anchor} — {title}")
        print(f"  LLM response:")
        
        # Parse and validate — check if suggested courses actually exist
        for line in raw.split("\n"):
            line = line.strip()
            if not line:
                continue
            # Try to extract course number
            match = re.search(r'(\d{1,2}\.\w{2,5})', line)
            if match:
                suggested_cid = match.group(1)
                exists = "✓" if suggested_cid in COURSES else "✗ NOT IN CATALOG"
                print(f"    {line[:80]}  [{exists}]")
            else:
                print(f"    {line[:80]}")
        
        llm_results[anchor] = raw
    except Exception as e:
        print(f"\n  Anchor: {anchor} — ERROR: {e}")

MODEL 4: LLM Direct

  Anchor: 6.3900 — Introduction to Machine Learning
  LLM response:
    Based on the description of 6.3900 — Introduction to Machine Learning, here are   [✓]
    1. 6.036 — Introduction to Machine Learning: This course is similar because it a  [✗ NOT IN CATALOG]
    2. 6.045 — Automata, Computability, and Complexity: Although primarily focused o  [✗ NOT IN CATALOG]
    3. 6.046 — Introduction to Algorithms: This course covers algorithms and data st  [✗ NOT IN CATALOG]
    4. 6.036J — Machine Learning: This course is a more advanced version of 6.036 an  [✗ NOT IN CATALOG]
    5. 15.097 — Machine Learning with Python: This course focuses on the practical i  [✓]

  Anchor: 18.06 — Linear Algebra
  LLM response:
    1. 18.700 — Linear Algebra:  Applications to Geometry and Physics: This course i  [✓]
    2. 18.702 — Linear Algebra:  Applications to Engineering and Computer Science: T  [✓]
    3. 18.712 — Linear Algebra and Differential Equations: This course is similar

---
## Comparison

For each anchor course, compare the top-5 results across all four models.

In [9]:
# ── Side-by-side comparison ──

print("SIDE-BY-SIDE COMPARISON")
print("=" * 100)

for anchor in ANCHORS:
    title = COURSES[anchor]["title"]
    print(f"\n{'━' * 100}")
    print(f"ANCHOR: {anchor} — {title}")
    print(f"{'━' * 100}")
    
    tfidf_top = [cid for cid, _ in tfidf_similar(anchor)]
    embed_top = [cid for cid, _ in embedding_similar(anchor)]
    hybrid_top = [cid for cid, _, _, _ in hybrid_similar(anchor)]
    
    print(f"\n  {'Rank':<6} {'TF-IDF':<25} {'Embeddings':<25} {'Hybrid':<25}")
    print(f"  {'-'*80}")
    for i in range(5):
        tf = f"{tfidf_top[i]} ({COURSES[tfidf_top[i]]['title'][:15]})" if i < len(tfidf_top) else ""
        em = f"{embed_top[i]} ({COURSES[embed_top[i]]['title'][:15]})" if i < len(embed_top) else ""
        hy = f"{hybrid_top[i]} ({COURSES[hybrid_top[i]]['title'][:15]})" if i < len(hybrid_top) else ""
        print(f"  #{i+1:<5} {tf:<25} {em:<25} {hy:<25}")
    
    # Agreement
    tf_set = set(tfidf_top[:5])
    em_set = set(embed_top[:5])
    hy_set = set(hybrid_top[:5])
    
    print(f"\n  Overlap: TF-IDF∩Embed={len(tf_set & em_set)}/5, "
          f"Embed∩Hybrid={len(em_set & hy_set)}/5, "
          f"All three={len(tf_set & em_set & hy_set)}/5")

SIDE-BY-SIDE COMPARISON

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ANCHOR: 6.3900 — Introduction to Machine Learning
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Rank   TF-IDF                    Embeddings                Hybrid                   
  --------------------------------------------------------------------------------
  #1     18.415 (Advanced Algori)  6.862 (Applied Machine)   6.862 (Applied Machine)  
  #2     6.5210 (Advanced Algori)  6.7900 (Machine Learnin)  6.7900 (Machine Learnin) 
  #3     6.7900 (Machine Learnin)  6.7250 (Optimization fo)  6.7250 (Optimization fo) 
  #4     6.8200 (Sensorimotor Le)  15.680 (Machine Learnin)  6.7910 (Statistical Lea) 
  #5     15.680 (Machine Learnin)  6.8200 (Sensorimotor Le)  6.8200 (Sensorimotor Le) 

  Overlap: TF-IDF∩Embed=3/5, Embed∩Hybrid=4/5, All three=2/5

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

---
## Model Selection:

**Selected Model: sentence embeddings**

The results show a clear winner. For the ML anchor (6.3900), embeddings and hybrid both surface the right courses — Applied Machine Learning, Machine Learning for Healthcare, Optimization for ML — while TF-IDF puts 18.415 (an algorithms course) at #1 based on keyword overlap rather than topical relevance. The economics anchor (14.01) is the starkest contrast: embeddings correctly finds economic analysis courses across departments (15.010, 15.011, 15.024), while TF-IDF returns entirely unrelated courses (Methods of Political Analysis, Integrated Chemical Engineering) that happen to share words like "methods" or "analysis." For linear algebra (18.06), all models agree on ES.1806 (the ES version of the same course) at #1, but embeddings and hybrid find matrix methods and numerical linear algebra courses while TF-IDF pulls in 2.087 (Engineering Math) on keyword match.

Embeddings and hybrid produce nearly identical results (4/5 overlap on all three anchors), meaning the structural bonus adds minimal value — the semantic signal from descriptions already captures department and topic relationships. The hybrid's only visible effect is promoting 6.7910 (Statistical Learning Theory) into the ML results over 15.680 (Machine Learning for Healthcare), which is a minor reordering. Given that the structural features add complexity without meaningful improvement, we select **sentence embeddings** as the course similarity model. It captures semantic relationships that TF-IDF misses entirely, runs fast enough to precompute for all courses, and the 4/5 overlap with hybrid confirms that text similarity alone is sufficient.